# 1. Creation of the df


## 1.1. Creation of Thermic_cool df

In [1]:
import os
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Alignment
from openpyxl import Workbook
from scipy.optimize import curve_fit

# Constants
TEMP_THRESHOLD = 32.7
MIN_TEMP = 15.0
# PTS_TEMP_RANGE = (32.6, 33.4)

# Chemin du dossier principal contenant les sous-dossiers pour chaque animal
folder_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2'

# Chemin du fichier animals_info.xlsx
info_file_path = os.path.join(folder_path, 'animals_info.xlsx')

# Chargement des informations sur les animaux
info_df = pd.read_excel(info_file_path)

# Assurez-vous que le nom des colonnes correspond bien
animal_column = 'Animal'
age_column = 'Age'
sexe_column = 'Sexe'
weight_column = 'Weight'

# Initialisation d'une liste pour stocker les DataFrames de chaque fichier
dataframes = []

# Fonction pour calculer le Zscore
def calculate_z_score_by_rec(df):
    """
    Calculer le Zscore pour chaque combinaison de 'rec' et 'animal' sur l'ensemble des trials.
    """
    df['Zscore'] = None  # Initialiser avec None

    # Grouper par 'rec' et 'animal'
    grouped = df.groupby(['rec', 'animal'])

    for (rec, animal), group in grouped:
        rec_mask = (df['rec'] == rec) & (df['animal'] == animal)

        # Calculer la moyenne et l'écart-type des valeurs de fluorescence pour chaque rec
        mean_fluo = df.loc[rec_mask, 'fluorescence thermo'].mean()
        std_fluo = df.loc[rec_mask, 'fluorescence thermo'].std()

        # Calculer le Zscore pour chaque ligne dans le groupe
        df.loc[rec_mask, 'Zscore'] = (df.loc[rec_mask, 'fluorescence thermo'] - mean_fluo) / std_fluo

    return df

# Fonction pour détecter les périodes "TS" et "TB"
def detect_ts_tb_periods(df):
    df['period'] = None
    period_number = 1
    i = 0

    while i < len(df):
        if df.loc[i, 'temperature'] < TEMP_THRESHOLD:
            start_idx = i
            while i < len(df) and df.loc[i, 'temperature'] < TEMP_THRESHOLD:
                i += 1

            if i - start_idx >= 30:
                min_temp = df.loc[start_idx:i, 'temperature'].min()
                if min_temp < MIN_TEMP:
                    end_idx = start_idx
                    while end_idx < i - 1 and df.loc[end_idx + 1, 'temperature'] <= df.loc[end_idx, 'temperature']:
                        end_idx += 1

                    df.loc[start_idx:end_idx, 'period'] = f'TS_{period_number}'
                    tb_start_idx = max(0, start_idx - 100)
                    df.loc[tb_start_idx:start_idx - 1, 'period'] = f'TB_{period_number}'

                    period_number += 1
                    i = end_idx + 1
                    continue
        i += 1

    return df


# Fonction pour détecter les périodes "PTS"
def detect_pts_periods(df):
    # Remplacer les valeurs manquantes de 'period' par 'None'
    df['period'] = df['period'].fillna('None')
    i = 0
    period_number = 1

    while i < len(df):
        # Détecter la fin de la période TS
        if 'TS_' in df.loc[i, 'period']:
            start_ts_idx = i
            while i < len(df) and 'TS_' in df.loc[i, 'period']:
                i += 1
            end_ts_idx = i - 1

            # Calculer la fin de la période PTS pour que TS + PTS = 101 lignes
            start_pts_idx = end_ts_idx + 1
            ts_length = end_ts_idx - start_ts_idx + 1
            pts_length = 101 - ts_length
            end_pts_idx = start_pts_idx + pts_length - 1

            # S'assurer que la période PTS ne dépasse pas la taille du DataFrame
            if end_pts_idx < len(df):
                # Appliquer 'PTS' uniquement aux lignes qui ne sont pas déjà 'TS'
                df.loc[start_pts_idx:end_pts_idx, 'period'] = df.loc[start_pts_idx:end_pts_idx, 'period'].apply(
                    lambda x: f'PTS_{period_number}' if 'None' in x else x
                )
                period_number += 1
                i = end_pts_idx + 1
            else:
                # Si la fin dépasse les limites du DataFrame, ajustez et arrêtez
                break
        else:
            i += 1

    return df


# Fonction pour détecter les périodes "ITST"
def detect_itst_periods(df):
    df['period'] = df['period'].fillna('None')
    itst_number = 1

    first_tb_idx = df[df['period'].str.contains('TB_', na=False)].index.min()
    if first_tb_idx > 0:
        df.loc[:first_tb_idx - 1, 'period'] = f'ITST_{itst_number}'
        itst_number += 1

    ts_tb_periods = df[df['period'].str.contains('TS_|TB_', na=False)]

    for i in range(len(ts_tb_periods) - 1):
        current_period = ts_tb_periods.iloc[i]
        next_period = ts_tb_periods.iloc[i + 1]

        if 'PTS_' in current_period['period'] and 'TB_' in next_period['period']:
            start_idx = current_period.name + 1
            end_idx = next_period.name - 1
            if end_idx > start_idx:
                df.loc[start_idx:end_idx, 'period'] = f'ITST_{itst_number}'
                itst_number += 1

    last_pts_idx = df[df['period'].str.contains('PTS_', na=False)].index.max()
    if last_pts_idx < len(df) - 1:
        df.loc[last_pts_idx + 1:, 'period'] = 'ITST_9'

    return df

# Nouvelle fonction pour définir les trials
def define_trials(df):
    df['trial'] = None
    trial_number = 1
    i = 0

    while i < len(df):
        if trial_number == 1:
            itst1 = df[df['period'] == 'ITST_1']
            tb1 = df[df['period'] == 'TB_1']
            ts1 = df[df['period'] == 'TS_1']
            pts1 = df[df['period'] == 'PTS_1']
            itst2 = df[df['period'] == 'ITST_2']

            trial_df = pd.concat([itst1, tb1, ts1, pts1, itst2])
            df.loc[trial_df.index, 'trial'] = f'Trial_{trial_number}'
            trial_number += 1
            i = trial_df.index[-1] + 1
        else:
            tb_n = df[df['period'] == f'TB_{trial_number}']
            ts_n = df[df['period'] == f'TS_{trial_number}']
            pts_n = df[df['period'] == f'PTS_{trial_number}']
            itst_nplus1 = df[df['period'] == f'ITST_{trial_number + 1}']

            trial_df = pd.concat([tb_n, ts_n, pts_n, itst_nplus1])
            df.loc[trial_df.index, 'trial'] = f'Trial_{trial_number}'
            trial_number += 1
            i = trial_df.index[-1] + 1

            if trial_number > 8:
                break

    return df

# Fonction pour calculer le Zscore normalisé par la médiane de TB pour chaque trial
def calculate_z_score_norm_by_trial(df):
    """
    Calculer le Zscore normalisé par la moyenne du Zscore de la période TB
    pour chaque 'trial', 'rec', et 'animal'.
    """
    df['Znorm'] = None  # Initialiser avec None

    # Grouper par 'trial', 'rec', et 'animal'
    grouped = df.groupby(['trial', 'rec', 'animal'])

    for (trial, rec, animal), group in grouped:
        trial_mask = (df['trial'] == trial) & (df['rec'] == rec) & (df['animal'] == animal)

        tb_mask = trial_mask & df['period'].str.contains('TB_', na=False)
        ts_pts_mask = trial_mask & df['period'].str.contains('TS_|PTS_', na=False)

        if not tb_mask.any():
            continue

        # Calculer la moyenne des Zscores pour la période TB
        tb_mean = df.loc[tb_mask, 'Zscore'].mean()

        # Appliquer la normalisation Zscore pour les périodes TB, TS et PTS du trial courant
        df.loc[tb_mask | ts_pts_mask, 'Znorm'] = df.loc[tb_mask | ts_pts_mask, 'Zscore'] - tb_mean

    return df

# Fonction pour calculer la colonne Stim_Time
def calculate_stim_time(df):
    df['Stim_Time'] = np.nan  # Initialiser la colonne Stim_Time avec des NaN

    for trial in df['trial'].unique():
        trial_df = df[df['trial'] == trial]

        # Gestion de TB_ période
        tb_period = trial_df[trial_df['period'].str.contains('TB_')]
        if not tb_period.empty:
            tb_start_idx = tb_period.index[0]
            tb_end_idx = tb_period.index[-1]

            tb_duration = tb_end_idx - tb_start_idx + 1
            tb_time_values = np.linspace(-100 * tb_duration, -100, tb_duration)

            df.loc[tb_start_idx:tb_end_idx, 'Stim_Time'] = tb_time_values

        # Gestion de TS_ et PTS_ périodes
        ts_period = trial_df[trial_df['period'].str.contains('TS_')]
        pts_period = trial_df[trial_df['period'].str.contains('PTS_')]

        if not ts_period.empty and not pts_period.empty:
            ts_start_idx = ts_period.index[0]
            pts_end_idx = pts_period.index[-1]

            df.loc[ts_start_idx, 'Stim_Time'] = 0  # TS_ commence à 0
            pts_duration = pts_end_idx - ts_start_idx
            pts_time_values = np.linspace(0, 100 * pts_duration, pts_duration + 1)

            df.loc[ts_start_idx:pts_end_idx, 'Stim_Time'] = pts_time_values

        # Les périodes ITST sont laissées en NaN
        itst_period = trial_df[trial_df['period'].str.contains('ITST_')]
        if not itst_period.empty:
            df.loc[itst_period.index, 'Stim_Time'] = np.nan

    return df

# Parcours de tous les sous-dossiers dans le dossier principal
for root, dirs, files in os.walk(folder_path):
    animal_name = os.path.basename(root)
    files.sort()

    for file in files:
        if file.startswith('~$'):
            continue

        if 'full_traces_all_masks_thermic' in file:
            condition_part = file.split('_full_traces_all_masks_thermic')[0]
            condition = condition_part.split('_')[0]
            recording_name = file.split('_full_traces_all_masks_thermic')[0]

            file_path = os.path.join(root, file)
            df = pd.read_excel(file_path)

            df_selected = df.iloc[:, [0, 2, 4]].copy()
            df_selected.columns = ['frame', 'temperature', 'fluorescence thermo']
            df_selected['time'] = df_selected['frame'] * 100
            df_selected['temperature'] = df_selected['temperature'].interpolate(method='linear', limit_direction='both')
            df_selected['rec'] = recording_name
            df_selected['animal'] = animal_name
            df_selected['age'] = None
            df_selected['sexe'] = None
            df_selected['weight'] = None
            df_selected['condition'] = condition
            df_selected['period'] = None
            df_selected['trial'] = None

            if animal_name in info_df[animal_column].values:
                animal_info = info_df[info_df[animal_column] == animal_name]
                if not animal_info.empty:
                    animal_info = animal_info.iloc[0]
                    df_selected['age'] = animal_info[age_column]
                    df_selected['sexe'] = animal_info[sexe_column]
                    df_selected['weight'] = animal_info[weight_column]
                else:
                    print(f"Aucune information trouvée pour l'animal: {animal_name}")
            else:
                print(f"Animal {animal_name} non trouvé dans 'animals_info.xlsx'")

            df_selected = calculate_z_score_by_rec(df_selected)
            df_selected = detect_ts_tb_periods(df_selected)
            df_selected = detect_pts_periods(df_selected)
            df_selected = detect_itst_periods(df_selected)
            df_selected = define_trials(df_selected)
            df_selected = calculate_z_score_norm_by_trial(df_selected)
            df_selected['time_sec'] = df_selected['time'] / 1000
            df_selected = calculate_stim_time(df_selected)  # Calculer Stim_Time ici

            # Réorganiser les colonnes pour insérer Stim_Time après time
            df_selected = df_selected[['animal', 'rec', 'age', 'sexe', 'weight', 'trial', 'period', 'condition', 'frame',
                                       'time', 'time_sec', 'Stim_Time', 'temperature', 'fluorescence thermo', 'Zscore', 'Znorm']]

            dataframes.append(df_selected)

# Combiner tous les DataFrames en un seul DataFrame final
final_df = pd.concat(dataframes, ignore_index=True)

# Chemin du fichier de sortie
output_file_path = os.path.join(folder_path, 'Df_thq_Zscore_classic.xlsx')

# Créer un nouveau classeur Excel et ajouter la feuille avec les données combinées
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    final_df.to_excel(writer, sheet_name='Master_sheet', index=False)

# Charger le fichier Excel pour ajuster les colonnes et ajouter le rapport
wb = load_workbook(output_file_path)
ws = wb.active

# Fonction pour appliquer l'alignement aux cellules
def center_cells(sheet):
    for row in sheet.iter_rows():
        for cell in row:
            cell.alignment = Alignment(horizontal='center', vertical='center')

# Ajuster la largeur des colonnes
for col in ws.columns:
    max_length = 0
    column = col[0].column_letter  # Colonne actuelle (A, B, C, etc.)
    for cell in col:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(cell.value)
        except:
            pass
    adjusted_width = (max_length + 2)
    ws.column_dimensions[column].width = adjusted_width

# Centrer les données dans la feuille active
center_cells(ws)

# Créer le rapport dans une nouvelle feuille
ws2 = wb.create_sheet(title="Report")

# Compter le nombre de lignes par enregistrement et par animal
report_df = final_df.groupby(['animal', 'rec']).size().reset_index(name='row_count')

# Ajouter le rapport à la nouvelle feuille
for r in dataframe_to_rows(report_df, index=False, header=True):
    ws2.append(r)

# Sauvegarder le fichier avec les modifications
wb.save(output_file_path)


#### Without ITST

In [2]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_Zscore_classic.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_Zscore_classic_stim.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où 'Stim_Time' n'est pas NaN
df_filtered = df.dropna(subset=['Stim_Time'])

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
df_filtered.to_excel(output_file_path, index=False)

print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")


Le DataFrame filtré a été sauvegardé sous G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_Zscore_classic_stim.xlsx


## 1.2. Creation of Thermic_Ure df

In [3]:
import os
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Alignment
from openpyxl import Workbook
from scipy.optimize import curve_fit

# Constants
TEMP_THRESHOLD = 32.7
MIN_TEMP = 15.0
# PTS_TEMP_RANGE = (32.6, 33.4)

# Chemin du dossier principal contenant les sous-dossiers pour chaque animal
folder_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2'

# Chemin du fichier animals_info.xlsx
info_file_path = os.path.join(folder_path, 'animals_info.xlsx')

# Chargement des informations sur les animaux
info_df = pd.read_excel(info_file_path)

# Assurez-vous que le nom des colonnes correspond bien
animal_column = 'Animal'
age_column = 'Age'
sexe_column = 'Sexe'
weight_column = 'Weight'

# Initialisation d'une liste pour stocker les DataFrames de chaque fichier
dataframes = []

# Fonction pour calculer le Zscore
def calculate_z_score_by_rec(df):
    """
    Calculer le Zscore pour chaque combinaison de 'rec' et 'animal' sur l'ensemble des trials.
    """
    df['Zscore'] = None  # Initialiser avec None

    # Grouper par 'rec' et 'animal'
    grouped = df.groupby(['rec', 'animal'])

    for (rec, animal), group in grouped:
        rec_mask = (df['rec'] == rec) & (df['animal'] == animal)

        # Calculer la moyenne et l'écart-type des valeurs de fluorescence pour chaque rec
        mean_fluo = df.loc[rec_mask, 'fluorescence thermo'].mean()
        std_fluo = df.loc[rec_mask, 'fluorescence thermo'].std()

        # Calculer le Zscore pour chaque ligne dans le groupe
        df.loc[rec_mask, 'Zscore'] = (df.loc[rec_mask, 'fluorescence thermo'] - mean_fluo) / std_fluo

    return df

# Fonction pour détecter les périodes "TS" et "TB"
def detect_ts_tb_periods(df):
    df['period'] = None
    period_number = 1
    i = 0

    while i < len(df):
        if df.loc[i, 'temperature'] < TEMP_THRESHOLD:
            start_idx = i
            while i < len(df) and df.loc[i, 'temperature'] < TEMP_THRESHOLD:
                i += 1

            if i - start_idx >= 30:
                min_temp = df.loc[start_idx:i, 'temperature'].min()
                if min_temp < MIN_TEMP:
                    end_idx = start_idx
                    while end_idx < i - 1 and df.loc[end_idx + 1, 'temperature'] <= df.loc[end_idx, 'temperature']:
                        end_idx += 1

                    df.loc[start_idx:end_idx, 'period'] = f'TS_{period_number}'
                    tb_start_idx = max(0, start_idx - 100)
                    df.loc[tb_start_idx:start_idx - 1, 'period'] = f'TB_{period_number}'

                    period_number += 1
                    i = end_idx + 1
                    continue
        i += 1

    return df


# Fonction pour détecter les périodes "PTS"
def detect_pts_periods(df):
    # Remplacer les valeurs manquantes de 'period' par 'None'
    df['period'] = df['period'].fillna('None')
    i = 0
    period_number = 1

    while i < len(df):
        # Détecter la fin de la période TS
        if 'TS_' in df.loc[i, 'period']:
            start_ts_idx = i
            while i < len(df) and 'TS_' in df.loc[i, 'period']:
                i += 1
            end_ts_idx = i - 1

            # Calculer la fin de la période PTS pour que TS + PTS = 101 lignes
            start_pts_idx = end_ts_idx + 1
            ts_length = end_ts_idx - start_ts_idx + 1
            pts_length = 101 - ts_length
            end_pts_idx = start_pts_idx + pts_length - 1

            # S'assurer que la période PTS ne dépasse pas la taille du DataFrame
            if end_pts_idx < len(df):
                # Appliquer 'PTS' uniquement aux lignes qui ne sont pas déjà 'TS'
                df.loc[start_pts_idx:end_pts_idx, 'period'] = df.loc[start_pts_idx:end_pts_idx, 'period'].apply(
                    lambda x: f'PTS_{period_number}' if 'None' in x else x
                )
                period_number += 1
                i = end_pts_idx + 1
            else:
                # Si la fin dépasse les limites du DataFrame, ajustez et arrêtez
                break
        else:
            i += 1

    return df


# Fonction pour détecter les périodes "ITST"
def detect_itst_periods(df):
    df['period'] = df['period'].fillna('None')
    itst_number = 1

    first_tb_idx = df[df['period'].str.contains('TB_', na=False)].index.min()
    if first_tb_idx > 0:
        df.loc[:first_tb_idx - 1, 'period'] = f'ITST_{itst_number}'
        itst_number += 1

    ts_tb_periods = df[df['period'].str.contains('TS_|TB_', na=False)]

    for i in range(len(ts_tb_periods) - 1):
        current_period = ts_tb_periods.iloc[i]
        next_period = ts_tb_periods.iloc[i + 1]

        if 'PTS_' in current_period['period'] and 'TB_' in next_period['period']:
            start_idx = current_period.name + 1
            end_idx = next_period.name - 1
            if end_idx > start_idx:
                df.loc[start_idx:end_idx, 'period'] = f'ITST_{itst_number}'
                itst_number += 1

    last_pts_idx = df[df['period'].str.contains('PTS_', na=False)].index.max()
    if last_pts_idx < len(df) - 1:
        df.loc[last_pts_idx + 1:, 'period'] = 'ITST_9'

    return df

# Nouvelle fonction pour définir les trials
def define_trials(df):
    df['trial'] = None
    trial_number = 1
    i = 0

    while i < len(df):
        if trial_number == 1:
            itst1 = df[df['period'] == 'ITST_1']
            tb1 = df[df['period'] == 'TB_1']
            ts1 = df[df['period'] == 'TS_1']
            pts1 = df[df['period'] == 'PTS_1']
            itst2 = df[df['period'] == 'ITST_2']

            trial_df = pd.concat([itst1, tb1, ts1, pts1, itst2])
            df.loc[trial_df.index, 'trial'] = f'Trial_{trial_number}'
            trial_number += 1
            i = trial_df.index[-1] + 1
        else:
            tb_n = df[df['period'] == f'TB_{trial_number}']
            ts_n = df[df['period'] == f'TS_{trial_number}']
            pts_n = df[df['period'] == f'PTS_{trial_number}']
            itst_nplus1 = df[df['period'] == f'ITST_{trial_number + 1}']

            trial_df = pd.concat([tb_n, ts_n, pts_n, itst_nplus1])
            df.loc[trial_df.index, 'trial'] = f'Trial_{trial_number}'
            trial_number += 1
            i = trial_df.index[-1] + 1

            if trial_number > 8:
                break

    return df

# Fonction pour calculer le Zscore normalisé par la médiane de TB pour chaque trial
def calculate_z_score_norm_by_trial(df):
    """
    Calculer le Zscore normalisé par la moyenne du Zscore de la période TB
    pour chaque 'trial', 'rec', et 'animal'.
    """
    df['Znorm'] = None  # Initialiser avec None

    # Grouper par 'trial', 'rec', et 'animal'
    grouped = df.groupby(['trial', 'rec', 'animal'])

    for (trial, rec, animal), group in grouped:
        trial_mask = (df['trial'] == trial) & (df['rec'] == rec) & (df['animal'] == animal)

        tb_mask = trial_mask & df['period'].str.contains('TB_', na=False)
        ts_pts_mask = trial_mask & df['period'].str.contains('TS_|PTS_', na=False)

        if not tb_mask.any():
            continue

        # Calculer la moyenne des Zscores pour la période TB
        tb_mean = df.loc[tb_mask, 'Zscore'].mean()

        # Appliquer la normalisation Zscore pour les périodes TB, TS et PTS du trial courant
        df.loc[tb_mask | ts_pts_mask, 'Znorm'] = df.loc[tb_mask | ts_pts_mask, 'Zscore'] - tb_mean

    return df

# Fonction pour calculer la colonne Stim_Time
def calculate_stim_time(df):
    df['Stim_Time'] = np.nan  # Initialiser la colonne Stim_Time avec des NaN

    for trial in df['trial'].unique():
        trial_df = df[df['trial'] == trial]

        # Gestion de TB_ période
        tb_period = trial_df[trial_df['period'].str.contains('TB_')]
        if not tb_period.empty:
            tb_start_idx = tb_period.index[0]
            tb_end_idx = tb_period.index[-1]

            tb_duration = tb_end_idx - tb_start_idx + 1
            tb_time_values = np.linspace(-100 * tb_duration, -100, tb_duration)

            df.loc[tb_start_idx:tb_end_idx, 'Stim_Time'] = tb_time_values

        # Gestion de TS_ et PTS_ périodes
        ts_period = trial_df[trial_df['period'].str.contains('TS_')]
        pts_period = trial_df[trial_df['period'].str.contains('PTS_')]

        if not ts_period.empty and not pts_period.empty:
            ts_start_idx = ts_period.index[0]
            pts_end_idx = pts_period.index[-1]

            df.loc[ts_start_idx, 'Stim_Time'] = 0  # TS_ commence à 0
            pts_duration = pts_end_idx - ts_start_idx
            pts_time_values = np.linspace(0, 100 * pts_duration, pts_duration + 1)

            df.loc[ts_start_idx:pts_end_idx, 'Stim_Time'] = pts_time_values

        # Les périodes ITST sont laissées en NaN
        itst_period = trial_df[trial_df['period'].str.contains('ITST_')]
        if not itst_period.empty:
            df.loc[itst_period.index, 'Stim_Time'] = np.nan

    return df

# Parcours de tous les sous-dossiers dans le dossier principal
for root, dirs, files in os.walk(folder_path):
    animal_name = os.path.basename(root)
    files.sort()

    for file in files:
        if file.startswith('~$'):
            continue

        if 'full_traces_all_masks_thq_ure' in file:
            condition_part = file.split('_full_traces_all_masks_thq_ure')[0]
            condition = condition_part.split('_')[0]
            recording_name = file.split('_full_traces_all_masks_thq_ure')[0]

            file_path = os.path.join(root, file)
            df = pd.read_excel(file_path)

            df_selected = df.iloc[:, [0, 2, 4]].copy()
            df_selected.columns = ['frame', 'temperature', 'fluorescence thermo']
            df_selected['time'] = df_selected['frame'] * 100
            df_selected['temperature'] = df_selected['temperature'].interpolate(method='linear', limit_direction='both')
            df_selected['rec'] = recording_name
            df_selected['animal'] = animal_name
            df_selected['age'] = None
            df_selected['sexe'] = None
            df_selected['weight'] = None
            df_selected['condition'] = condition
            df_selected['period'] = None
            df_selected['trial'] = None

            if animal_name in info_df[animal_column].values:
                animal_info = info_df[info_df[animal_column] == animal_name]
                if not animal_info.empty:
                    animal_info = animal_info.iloc[0]
                    df_selected['age'] = animal_info[age_column]
                    df_selected['sexe'] = animal_info[sexe_column]
                    df_selected['weight'] = animal_info[weight_column]
                else:
                    print(f"Aucune information trouvée pour l'animal: {animal_name}")
            else:
                print(f"Animal {animal_name} non trouvé dans 'animals_info.xlsx'")

            df_selected = calculate_z_score_by_rec(df_selected)
            df_selected = detect_ts_tb_periods(df_selected)
            df_selected = detect_pts_periods(df_selected)
            df_selected = detect_itst_periods(df_selected)
            df_selected = define_trials(df_selected)
            df_selected = calculate_z_score_norm_by_trial(df_selected)
            df_selected['time_sec'] = df_selected['time'] / 1000
            df_selected = calculate_stim_time(df_selected)  # Calculer Stim_Time ici

            # Réorganiser les colonnes pour insérer Stim_Time après time
            df_selected = df_selected[['animal', 'rec', 'age', 'sexe', 'weight', 'trial', 'period', 'condition', 'frame',
                                       'time', 'time_sec', 'Stim_Time', 'temperature', 'fluorescence thermo', 'Zscore', 'Znorm']]

            dataframes.append(df_selected)

# Combiner tous les DataFrames en un seul DataFrame final
final_df = pd.concat(dataframes, ignore_index=True)

# Chemin du fichier de sortie
output_file_path = os.path.join(folder_path, 'Df_thq_ure_Zscore_classic.xlsx')

# Créer un nouveau classeur Excel et ajouter la feuille avec les données combinées
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    final_df.to_excel(writer, sheet_name='Master_sheet', index=False)

# Charger le fichier Excel pour ajuster les colonnes et ajouter le rapport
wb = load_workbook(output_file_path)
ws = wb.active

# Fonction pour appliquer l'alignement aux cellules
def center_cells(sheet):
    for row in sheet.iter_rows():
        for cell in row:
            cell.alignment = Alignment(horizontal='center', vertical='center')

# Ajuster la largeur des colonnes
for col in ws.columns:
    max_length = 0
    column = col[0].column_letter  # Colonne actuelle (A, B, C, etc.)
    for cell in col:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(cell.value)
        except:
            pass
    adjusted_width = (max_length + 2)
    ws.column_dimensions[column].width = adjusted_width

# Centrer les données dans la feuille active
center_cells(ws)

# Créer le rapport dans une nouvelle feuille
ws2 = wb.create_sheet(title="Report")

# Compter le nombre de lignes par enregistrement et par animal
report_df = final_df.groupby(['animal', 'rec']).size().reset_index(name='row_count')

# Ajouter le rapport à la nouvelle feuille
for r in dataframe_to_rows(report_df, index=False, header=True):
    ws2.append(r)

# Sauvegarder le fichier avec les modifications
wb.save(output_file_path)


#### Without ITST

In [4]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_ure_Zscore_classic.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_ure_Zscore_classic_stim.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où 'Stim_Time' n'est pas NaN
df_filtered = df.dropna(subset=['Stim_Time'])

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
df_filtered.to_excel(output_file_path, index=False)

print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")


Le DataFrame filtré a été sauvegardé sous G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_ure_Zscore_classic_stim.xlsx


# 2. Df creation for each animal with Mean-SEM

## 2.1. Df for each animal with Mean-SEM - Thq

In [5]:
import pandas as pd
import os

# Chemin du fichier Excel à modifier
file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_Zscore_classic.xlsx'

# Charger le fichier Excel dans un DataFrame
df = pd.read_excel(file_path)

# Obtenir la liste des valeurs uniques dans la colonne 'animal'
unique_animals = df['animal'].unique()

# Extraire le nom du fichier sans extension
file_name, _ = os.path.splitext(os.path.basename(file_path))

# Répertoire principal de sauvegarde
main_save_directory = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/'
# Créer un sous-répertoire pour les fichiers filtrés
filtered_save_directory = os.path.join(main_save_directory, 'Df_Thq_by_animal')
if not os.path.exists(filtered_save_directory):
    os.makedirs(filtered_save_directory)

# Traiter chaque animal individuellement
for animal in unique_animals:
    # Filtrer le DataFrame pour ne garder que les lignes où 'animal' est l'animal actuel
    df_filtered = df[df['animal'] == animal]

    # Créer une chaîne pour le nom de l'animal à garder
    animals_str = animal

    # Créer le chemin du fichier Excel modifié pour l'animal
    output_file_path = os.path.join(filtered_save_directory, f'{file_name}_{animals_str}.xlsx')

    # Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
    df_filtered.to_excel(output_file_path, index=False)

    print(f"Le DataFrame filtré pour l'animal '{animal}' a été sauvegardé avec succès dans : {output_file_path}")


Le DataFrame filtré pour l'animal '2023.10.06' a été sauvegardé avec succès dans : G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_by_animal\Df_thq_Zscore_classic_2023.10.06.xlsx
Le DataFrame filtré pour l'animal '2023.10.07' a été sauvegardé avec succès dans : G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_by_animal\Df_thq_Zscore_classic_2023.10.07.xlsx
Le DataFrame filtré pour l'animal '2023.10.17' a été sauvegardé avec succès dans : G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_by_animal\Df_thq_Zscore_classic_2023.10.17.xlsx


## 2.2. Df for each animal with Mean-SEM - Thq_ure

In [6]:
import pandas as pd
import os

# Chemin du fichier Excel à modifier
file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_ure_Zscore_classic.xlsx'

# Charger le fichier Excel dans un DataFrame
df = pd.read_excel(file_path)

# Obtenir la liste des valeurs uniques dans la colonne 'animal'
unique_animals = df['animal'].unique()

# Extraire le nom du fichier sans extension
file_name, _ = os.path.splitext(os.path.basename(file_path))

# Répertoire principal de sauvegarde
main_save_directory = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/'
# Créer un sous-répertoire pour les fichiers filtrés
filtered_save_directory = os.path.join(main_save_directory, 'Df_Thq_ure_by_animal')
if not os.path.exists(filtered_save_directory):
    os.makedirs(filtered_save_directory)

# Traiter chaque animal individuellement
for animal in unique_animals:
    # Filtrer le DataFrame pour ne garder que les lignes où 'animal' est l'animal actuel
    df_filtered = df[df['animal'] == animal]

    # Créer une chaîne pour le nom de l'animal à garder
    animals_str = animal

    # Créer le chemin du fichier Excel modifié pour l'animal
    output_file_path = os.path.join(filtered_save_directory, f'{file_name}_{animals_str}.xlsx')

    # Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
    df_filtered.to_excel(output_file_path, index=False)

    print(f"Le DataFrame filtré pour l'animal '{animal}' a été sauvegardé avec succès dans : {output_file_path}")


Le DataFrame filtré pour l'animal '2023.10.06' a été sauvegardé avec succès dans : G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_ure_by_animal\Df_thq_ure_Zscore_classic_2023.10.06.xlsx
Le DataFrame filtré pour l'animal '2023.10.07' a été sauvegardé avec succès dans : G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_ure_by_animal\Df_thq_ure_Zscore_classic_2023.10.07.xlsx
Le DataFrame filtré pour l'animal '2023.10.17' a été sauvegardé avec succès dans : G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_ure_by_animal\Df_thq_ure_Zscore_classic_2023.10.17.xlsx


#

#

# 3. Creation des df "Trial_MeanSEM"

## 3.1. Df Thq_Trial_MeanSEM"

In [7]:
import pandas as pd
import numpy as np
import os
import glob

# Répertoires de sauvegarde
input_directory = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_by_animal/'
main_output_directory = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/'
output_directory = os.path.join(main_output_directory, 'Df_Thq_Trial_MeanSEM')

# Créer le répertoire de sortie s'il n'existe pas
if not os.path.exists(output_directory):
    os.makedirs(output_directory)


# Fonction pour détecter les périodes TS, TB, et PTS
def detect_periods(df):
    df['period'] = None
    i = 0
    while i < len(df):
        if df.iloc[i]['Mean_temperature'] < TEMP_THRESHOLD:
            start_idx = i
            while i < len(df) and df.iloc[i]['Mean_temperature'] < TEMP_THRESHOLD:
                i += 1
            end_idx = i

            if end_idx - start_idx >= 30:
                min_temp_idx = df.iloc[start_idx:end_idx]['Mean_temperature'].idxmin()
                min_temp = df.iloc[min_temp_idx]['Mean_temperature']

                if min_temp < MIN_TEMP:
                    # Marquer la période TS de start_idx jusqu'à la température minimale
                    df.loc[start_idx:min_temp_idx, 'period'] = 'TS'

                    # Marquer la période TB avant TS
                    tb_start_idx = max(0, start_idx - 100)
                    df.loc[tb_start_idx:start_idx - 1, 'period'] = 'TB'

                    # Marquer toutes les lignes après TS comme PTS
                    df.loc[min_temp_idx + 1:, 'period'] = 'PTS'
                break  # Sortir après avoir marqué TS et PTS

        i += 1

    return df

# Définir les seuils de température
TEMP_THRESHOLD = 32.7
MIN_TEMP = 15.0

# Trouver tous les fichiers correspondant au modèle
file_pattern = os.path.join(input_directory, 'Df_thq_Zscore_classic_*.xlsx')
files = glob.glob(file_pattern)

print(f"Fichiers trouvés : {files}")

for file_path in files:
    # Charger le fichier Excel dans un DataFrame
    df = pd.read_excel(file_path)

    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.strip()

    # Définir les colonnes nécessaires
    required_columns = ['trial', 'Stim_Time', 'Znorm', 'temperature']
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        print(f"Colonnes manquantes : {missing_columns}")
    else:
        # Filtrer les colonnes utiles
        df_filtered = df[required_columns]

        # Définir les valeurs de Stim_Time à considérer (de -10000 à 9300 avec un écart de 100)
        stim_time_range = range(-10000, 10100, 100)

        # Grouper par trial et Stim_Time, puis calculer la moyenne et l'écart-type du Znorm
        df_grouped = df_filtered[df_filtered['Stim_Time'].isin(stim_time_range)].groupby('Stim_Time').agg({
            'Znorm': ['mean', 'sem'],
            'temperature': 'mean'
        }).reset_index()

        # Aplatir les colonnes groupées
        df_grouped.columns = ['Stim_Time', 'Mean_Znorm', 'SEM_Znorm', 'Mean_temperature']


        # Détecter les périodes TS, TB, et PTS dans le DataFrame groupé
        df_grouped = detect_periods(df_grouped)
 

        # Créer le chemin de sortie avec le suffixe '_trial_mean.xlsx'
        base_name = os.path.splitext(os.path.basename(file_path))[0]
        output_file_path = os.path.join(output_directory, f'{base_name}_trial_mean_SEM.xlsx')

        # Sauvegarder le DataFrame final avec les périodes détectées
        df_grouped.to_excel(output_file_path, index=False)
        print(f"Le DataFrame final pour '{file_path}' a été sauvegardé avec succès dans : {output_file_path}")


Fichiers trouvés : ['G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_by_animal\\Df_thq_Zscore_classic_2023.10.06.xlsx', 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_by_animal\\Df_thq_Zscore_classic_2023.10.07.xlsx', 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_by_animal\\Df_thq_Zscore_classic_2023.10.17.xlsx']
Le DataFrame final pour 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_by_animal\Df_thq_Zscore_classic_2023.10.06.xlsx' a été sauvegardé avec succès dans : G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_Trial_MeanSEM\Df_thq_Zscore_classic_2023.10.06_trial_mean_SEM.xlsx
Le DataFrame final pour 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_by_animal\Df_thq_Zscore_classic_2023.10.07.xlsx' a été sauvegardé avec succès dans : G:/PhD/Ex

## 3.2. Df Thq_ure_Trial_MeanSEM"

In [8]:
import pandas as pd
import numpy as np
import os
import glob

# Répertoires de sauvegarde
input_directory = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_ure_by_animal/'
main_output_directory = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/'
output_directory = os.path.join(main_output_directory, 'Df_Thq_ure_Trial_MeanSEM')

# Créer le répertoire de sortie s'il n'existe pas
if not os.path.exists(output_directory):
    os.makedirs(output_directory)


# Fonction pour détecter les périodes TS, TB, et PTS
def detect_periods(df):
    df['period'] = None
    i = 0
    while i < len(df):
        if df.iloc[i]['Mean_temperature'] < TEMP_THRESHOLD:
            start_idx = i
            while i < len(df) and df.iloc[i]['Mean_temperature'] < TEMP_THRESHOLD:
                i += 1
            end_idx = i

            if end_idx - start_idx >= 30:
                min_temp_idx = df.iloc[start_idx:end_idx]['Mean_temperature'].idxmin()
                min_temp = df.iloc[min_temp_idx]['Mean_temperature']

                if min_temp < MIN_TEMP:
                    # Marquer la période TS de start_idx jusqu'à la température minimale
                    df.loc[start_idx:min_temp_idx, 'period'] = 'TS'

                    # Marquer la période TB avant TS
                    tb_start_idx = max(0, start_idx - 100)
                    df.loc[tb_start_idx:start_idx - 1, 'period'] = 'TB'

                    # Marquer toutes les lignes après TS comme PTS
                    df.loc[min_temp_idx + 1:, 'period'] = 'PTS'
                break  # Sortir après avoir marqué TS et PTS

        i += 1

    return df

# Définir les seuils de température
TEMP_THRESHOLD = 32.7
MIN_TEMP = 15.0

# Trouver tous les fichiers correspondant au modèle
file_pattern = os.path.join(input_directory, 'Df_thq_ure_Zscore_classic_*.xlsx')
files = glob.glob(file_pattern)

print(f"Fichiers trouvés : {files}")

for file_path in files:
    # Charger le fichier Excel dans un DataFrame
    df = pd.read_excel(file_path)

    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.strip()

    # Définir les colonnes nécessaires
    required_columns = ['trial', 'Stim_Time', 'Znorm', 'temperature']
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        print(f"Colonnes manquantes : {missing_columns}")
    else:
        # Filtrer les colonnes utiles
        df_filtered = df[required_columns]

        # Définir les valeurs de Stim_Time à considérer (de -10000 à 9300 avec un écart de 100)
        stim_time_range = range(-10000, 10100, 100)

        # Grouper par trial et Stim_Time, puis calculer la moyenne et l'écart-type du Znorm
        df_grouped = df_filtered[df_filtered['Stim_Time'].isin(stim_time_range)].groupby('Stim_Time').agg({
            'Znorm': ['mean', 'sem'],
            'temperature': 'mean'
        }).reset_index()

        # Aplatir les colonnes groupées
        df_grouped.columns = ['Stim_Time', 'Mean_Znorm', 'SEM_Znorm', 'Mean_temperature']


        # Détecter les périodes TS, TB, et PTS dans le DataFrame groupé
        df_grouped = detect_periods(df_grouped)
 

        # Créer le chemin de sortie avec le suffixe '_trial_mean.xlsx'
        base_name = os.path.splitext(os.path.basename(file_path))[0]
        output_file_path = os.path.join(output_directory, f'{base_name}_trial_mean_SEM.xlsx')

        # Sauvegarder le DataFrame final avec les périodes détectées
        df_grouped.to_excel(output_file_path, index=False)
        print(f"Le DataFrame final pour '{file_path}' a été sauvegardé avec succès dans : {output_file_path}")


Fichiers trouvés : ['G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_ure_by_animal\\Df_thq_ure_Zscore_classic_2023.10.06.xlsx', 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_ure_by_animal\\Df_thq_ure_Zscore_classic_2023.10.07.xlsx', 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_ure_by_animal\\Df_thq_ure_Zscore_classic_2023.10.17.xlsx']
Le DataFrame final pour 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_ure_by_animal\Df_thq_ure_Zscore_classic_2023.10.06.xlsx' a été sauvegardé avec succès dans : G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_ure_Trial_MeanSEM\Df_thq_ure_Zscore_classic_2023.10.06_trial_mean_SEM.xlsx
Le DataFrame final pour 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_Thq_ure_by_animal\Df_thq_ure_Zscore_classic_2023.10.07.xls

#

#

# 4. DF construction for the AUC

### Directory for quantifications

In [9]:
import os

# Définir le chemin de base et le répertoire à créer
base_dir = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2"
output_dir = os.path.join(base_dir, "Quantif_by_Periods")

# Vérifier si le répertoire existe, sinon le créer
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Le répertoire '{output_dir}' a été créé avec succès.")
else:
    print(f"Le répertoire '{output_dir}' existe déjà.")

# For ploting
# Définir le chemin de base et le répertoire à créer
base_dir = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2"
output_dir = os.path.join(base_dir, "Plot_Quantif_by_Periods")

# Vérifier si le répertoire existe, sinon le créer
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Le répertoire '{output_dir}' a été créé avec succès.")
else:
    print(f"Le répertoire '{output_dir}' existe déjà.")

Le répertoire 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2\Quantif_by_Periods' a été créé avec succès.
Le répertoire 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2\Plot_Quantif_by_Periods' a été créé avec succès.


## 4.1. For Thq stim

In [10]:
import pandas as pd
import numpy as np
from scipy.integrate import trapezoid

# Charger le dataset depuis le fichier Excel
file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_Zscore_classic_stim.xlsx"
df = pd.read_excel(file_path)

# Assurez-vous que la colonne 'Znorm' est de type numérique
df['Znorm'] = pd.to_numeric(df['Znorm'], errors='coerce')

# Extraire les essais uniques (en fonction de 'animal', 'rec', 'trial')
trials_unique = df[['animal', 'rec', 'trial', 'age', 'sexe', 'weight']].drop_duplicates().reset_index(drop=True)

# Créer une liste pour stocker les résultats
results = []

# Fonction pour calculer l'aire sous la courbe (AUC)
def calculate_auc(zscore_values, time_values):
    # Calculer l'aire au-dessus et en dessous de la baseline (0)
    auc_positive = trapezoid(zscore_values[zscore_values > 0], x=time_values[zscore_values > 0])
    auc_negative = trapezoid(zscore_values[zscore_values < 0], x=time_values[zscore_values < 0])
    total_auc = auc_positive + auc_negative
    return total_auc

# Itérer sur chaque essai unique
for _, trial_row in trials_unique.iterrows():
    # Filtrer le DataFrame pour obtenir les données de cet essai unique
    trial_data = df[(df['animal'] == trial_row['animal']) & 
                    (df['rec'] == trial_row['rec']) & 
                    (df['trial'] == trial_row['trial'])]
    
    # Grouper les données filtrées par période
    grouped = trial_data.groupby('period')
    
    # Itérer sur chaque période ('TS', 'TB', 'PTS')
    for period, group in grouped:
        # Extraire les valeurs de Zscore
        zscore_values = group['Znorm'].values
        time_values = group['time_sec'].values  # Assuming you have a time variable
        
        # Calculer l'aire sous la courbe (AUC)
        total_auc = calculate_auc(zscore_values, time_values)
        
        # Compter le nombre de lignes dans chaque période
        num_lines = len(group)
        
        # Calculer la durée de la période en ms et en s
        duration_ms = num_lines * 100  # Durée en ms
        duration_s = duration_ms / 1000  # Durée en s
        
        # Normaliser l'aire par la durée de la période (en secondes)
        auc_per_duration = total_auc / duration_s if duration_s > 0 else 0
        
        # Multiplication de la valeur de l'AUC pour visualisation
        auc10_per_duration = auc_per_duration * 10

        # Ajouter les résultats à la liste
        results.append({
            'Animal': trial_row['animal'],
            'Rec': trial_row['rec'],
            'Trial': trial_row['trial'],
            'Period': period,
            'AUC_Znorm_persec': auc10_per_duration,
            'Period_duration': duration_ms,  # Durée de la période en ms
            'Normalization_value': duration_s,  # Valeur utilisée pour normaliser en secondes
            'Age': trial_row['age'],
            'Sexe': trial_row['sexe'],
            'Weight': trial_row['weight']
        })

# Convertir la liste de résultats en DataFrame
df_auc_values = pd.DataFrame(results)

# Définir le chemin où sauvegarder le fichier Excel
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq.xlsx"

# Sauvegarder le DataFrame dans un fichier Excel
df_auc_values.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les données ont été sauvegardées dans {output_file_path}")


Les données ont été sauvegardées dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq.xlsx


### Supression des Periods index for stat on periods

In [11]:
import pandas as pd

# Charger le fichier Excel
file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq.xlsx"
df = pd.read_excel(file_path)

# Vérifier les colonnes disponibles
print(df.columns)

# Supposons que la colonne contenant les périodes s'appelle 'Period'
# Extraire les valeurs de 'Period' sans les suffixes '_index'
df['Period'] = df['Period'].str.extract(r'([A-Z]+)')[0]

# # Créer une nouvelle colonne 'Period_index' avec les indices
# df['Period_index'] = df['Period'].str.extract(r'_(\d+)')[0]

# Supprimer l'indice de la colonne 'Period'
# En utilisant str.replace pour supprimer l'indice dans 'Period'
df['Period'] = df['Period'].str.replace(r'_(\d+)', '', regex=True)

# Renommer le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq_wo_index.xlsx"
df.to_excel(output_file_path, index=False)

print(f"Le fichier a été enregistré sous : {output_file_path}")


Index(['Animal', 'Rec', 'Trial', 'Period', 'AUC_Znorm_persec',
       'Period_duration', 'Normalization_value', 'Age', 'Sexe', 'Weight'],
      dtype='object')
Le fichier a été enregistré sous : G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq_wo_index.xlsx


### Moyennage par animaux

In [12]:
import pandas as pd

# Charger le DataFrame contenant les valeurs d'AUC
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq_wo_index.xlsx"
df_auc_values = pd.read_excel(input_file_path)

# Calculer la moyenne des AUC par période, animal, sexe et âge
mean_auc_by_animal = df_auc_values.groupby(['Animal', 'Sexe', 'Age', 'Period'], as_index=False)['AUC_Znorm_persec'].mean()

# Définir le chemin où sauvegarder le fichier Excel des moyennes
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq_animal.xlsx"

# Sauvegarder le DataFrame des moyennes dans un fichier Excel
mean_auc_by_animal.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les moyennes d'AUC par période, sexe et âge ont été sauvegardées dans {output_file_path}")


Les moyennes d'AUC par période, sexe et âge ont été sauvegardées dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq_animal.xlsx


### Récupération des Data par animal pour period TS

In [13]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq_animal.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_thq_TS_only_animal.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où la colonne 'period' contient 'TS' (partout dans la cellule)
df_filtré = df[df['Period'].str.startswith("TS", na=False)]

# Vérifier le nombre de lignes filtrées
print(f"Nombre de lignes filtrées : {len(df_filtré)}")

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
if not df_filtré.empty:
    df_filtré.to_excel(output_file_path, index=False)
    print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")
else:
    print("Aucune ligne correspondante pour 'TS' trouvée.")

Nombre de lignes filtrées : 3
Le DataFrame filtré a été sauvegardé sous G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_thq_TS_only_animal.xlsx


## 4.2. For thq_ure

In [14]:
import pandas as pd
import numpy as np
from scipy.integrate import trapezoid

# Charger le dataset depuis le fichier Excel
file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_ure_Zscore_classic_stim.xlsx"
df = pd.read_excel(file_path)

# Assurez-vous que la colonne 'Znorm' est de type numérique
df['Znorm'] = pd.to_numeric(df['Znorm'], errors='coerce')

# Extraire les essais uniques (en fonction de 'animal', 'rec', 'trial')
trials_unique = df[['animal', 'rec', 'trial', 'age', 'sexe', 'weight']].drop_duplicates().reset_index(drop=True)

# Créer une liste pour stocker les résultats
results = []

# Fonction pour calculer l'aire sous la courbe (AUC)
def calculate_auc(zscore_values, time_values):
    # Calculer l'aire au-dessus et en dessous de la baseline (0)
    auc_positive = trapezoid(zscore_values[zscore_values > 0], x=time_values[zscore_values > 0])
    auc_negative = trapezoid(zscore_values[zscore_values < 0], x=time_values[zscore_values < 0])
    total_auc = auc_positive + auc_negative
    return total_auc

# Itérer sur chaque essai unique
for _, trial_row in trials_unique.iterrows():
    # Filtrer le DataFrame pour obtenir les données de cet essai unique
    trial_data = df[(df['animal'] == trial_row['animal']) & 
                    (df['rec'] == trial_row['rec']) & 
                    (df['trial'] == trial_row['trial'])]
    
    # Grouper les données filtrées par période
    grouped = trial_data.groupby('period')
    
    # Itérer sur chaque période ('TS', 'TB', 'PTS')
    for period, group in grouped:
        # Extraire les valeurs de Zscore
        zscore_values = group['Znorm'].values
        time_values = group['time_sec'].values  # Assuming you have a time variable
        
        # Calculer l'aire sous la courbe (AUC)
        total_auc = calculate_auc(zscore_values, time_values)
        
        # Compter le nombre de lignes dans chaque période
        num_lines = len(group)
        
        # Calculer la durée de la période en ms et en s
        duration_ms = num_lines * 100  # Durée en ms
        duration_s = duration_ms / 1000  # Durée en s
        
        # Normaliser l'aire par la durée de la période (en secondes)
        auc_per_duration = total_auc / duration_s if duration_s > 0 else 0
        
        # Multiplication de la valeur de l'AUC pour visualisation
        auc10_per_duration = auc_per_duration * 10

        # Ajouter les résultats à la liste
        results.append({
            'Animal': trial_row['animal'],
            'Rec': trial_row['rec'],
            'Trial': trial_row['trial'],
            'Period': period,
            'AUC_Znorm_persec': auc10_per_duration,
            'Period_duration': duration_ms,  # Durée de la période en ms
            'Normalization_value': duration_s,  # Valeur utilisée pour normaliser en secondes
            'Age': trial_row['age'],
            'Sexe': trial_row['sexe'],
            'Weight': trial_row['weight']
        })

# Convertir la liste de résultats en DataFrame
df_auc_values = pd.DataFrame(results)

# Définir le chemin où sauvegarder le fichier Excel
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq_ure.xlsx"

# Sauvegarder le DataFrame dans un fichier Excel
df_auc_values.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les données ont été sauvegardées dans {output_file_path}")


Les données ont été sauvegardées dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq_ure.xlsx


### Supression des Periods index for stat on periods

In [15]:
import pandas as pd

# Charger le fichier Excel
file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq_ure.xlsx"
df = pd.read_excel(file_path)

# Vérifier les colonnes disponibles
print(df.columns)

# Supposons que la colonne contenant les périodes s'appelle 'Period'
# Extraire les valeurs de 'Period' sans les suffixes '_index'
df['Period'] = df['Period'].str.extract(r'([A-Z]+)')[0]

# # Créer une nouvelle colonne 'Period_index' avec les indices
# df['Period_index'] = df['Period'].str.extract(r'_(\d+)')[0]

# Supprimer l'indice de la colonne 'Period'
# En utilisant str.replace pour supprimer l'indice dans 'Period'
df['Period'] = df['Period'].str.replace(r'_(\d+)', '', regex=True)

# Renommer le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq_ure_wo_index.xlsx"
df.to_excel(output_file_path, index=False)

print(f"Le fichier a été enregistré sous : {output_file_path}")


Index(['Animal', 'Rec', 'Trial', 'Period', 'AUC_Znorm_persec',
       'Period_duration', 'Normalization_value', 'Age', 'Sexe', 'Weight'],
      dtype='object')
Le fichier a été enregistré sous : G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq_ure_wo_index.xlsx


### Moyennage par animaux

In [16]:
import pandas as pd

# Charger le DataFrame contenant les valeurs d'AUC
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq_ure_wo_index.xlsx"
df_auc_values = pd.read_excel(input_file_path)

# Calculer la moyenne des AUC par période, animal, sexe et âge
mean_auc_by_animal = df_auc_values.groupby(['Animal', 'Sexe', 'Age', 'Period'], as_index=False)['AUC_Znorm_persec'].mean()

# Définir le chemin où sauvegarder le fichier Excel des moyennes
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq_ure_animal.xlsx"

# Sauvegarder le DataFrame des moyennes dans un fichier Excel
mean_auc_by_animal.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les moyennes d'AUC par période, sexe et âge ont été sauvegardées dans {output_file_path}")


Les moyennes d'AUC par période, sexe et âge ont été sauvegardées dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq_ure_animal.xlsx


### Récupération des Data par animal pour period TS

In [17]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_by_Period_thq_ure_animal.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_thq_ure_TS_only_animal.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où la colonne 'period' contient 'TS' (partout dans la cellule)
df_filtré = df[df['Period'].str.startswith("TS", na=False)]

# Vérifier le nombre de lignes filtrées
print(f"Nombre de lignes filtrées : {len(df_filtré)}")

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
if not df_filtré.empty:
    df_filtré.to_excel(output_file_path, index=False)
    print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")
else:
    print("Aucune ligne correspondante pour 'TS' trouvée.")

Nombre de lignes filtrées : 3
Le DataFrame filtré a été sauvegardé sous G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/AUC_persec_Znorm_thq_ure_TS_only_animal.xlsx


#

#

# 5. DF construction for the Max

## 5.1. For thq stim

In [18]:
import pandas as pd
import numpy as np

# Charger le dataset depuis le fichier Excel
file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_Zscore_classic_stim.xlsx"
df = pd.read_excel(file_path)

# Assurez-vous que la colonne 'Znorm' est de type numérique
df['Znorm'] = pd.to_numeric(df['Znorm'], errors='coerce')

# Extraire les trials uniques (en fonction de 'animal', 'Trial', 'trial')
# On inclut aussi les colonnes 'age', 'sexe', et 'weight' dans les données uniques
trials_unique = df[['animal', 'rec', 'trial', 'age', 'sexe', 'weight']].drop_duplicates().reset_index(drop=True)

# Créer une liste pour stocker les résultats
results = []

# Itérer sur chaque trial unique
for _, trial_row in trials_unique.iterrows():
    # Filtrer le DataFrame pour obtenir les données de ce trial unique
    trial_data = df[(df['animal'] == trial_row['animal']) & 
                    (df['rec'] == trial_row['rec']) & 
                    (df['trial'] == trial_row['trial'])]
    
    # Grouper les données filtrées par période
    grouped = trial_data.groupby('period')
    
    # Itérer sur chaque période ('TS', 'TB', 'PTS')
    for period, group in grouped:
        # Trouver la ligne avec la valeur maximale de Znorm
        max_row = group.loc[group['Znorm'].idxmax()]
        
        # Extraire les informations nécessaires
        Max_Znorm = max_row['Znorm']
        associated_temp = max_row['temperature']
        stim_time = max_row['Stim_Time']
        
        # Ajouter les résultats à la liste
        results.append({
            'Animal': trial_row['animal'],
            'Rec': trial_row['rec'],
            'Trial': trial_row['trial'],
            'Period': period,
            'Max_Znorm': Max_Znorm,
            'Associated Temperature': associated_temp,
            'Stim Time': stim_time,
            'Age': trial_row['age'],
            'Sexe': trial_row['sexe'],
            'Weight': trial_row['weight']
        })

# Convertir la liste de résultats en DataFrame
df_max_values = pd.DataFrame(results)

# Définir le chemin où sauvegarder le fichier Excel
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq.xlsx"

# Sauvegarder le DataFrame dans un fichier Excel
df_max_values.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les données ont été sauvegardées dans {output_file_path}")


Les données ont été sauvegardées dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq.xlsx


### Supression des Periods index for stat on periods

In [19]:
import pandas as pd

# Charger le fichier Excel
file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq.xlsx"
df = pd.read_excel(file_path)

# Vérifier les colonnes disponibles
print(df.columns)

# Extraire les valeurs de 'Period' sans les suffixes '_index'
df['Period'] = df['Period'].str.extract(r'([A-Z]+)')[0]

# En utilisant str.replace pour supprimer l'indice dans 'Period'
df['Period'] = df['Period'].str.replace(r'_(\d+)', '', regex=True)

# Renommer le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq_wo_index.xlsx"
df.to_excel(output_file_path, index=False)

print(f"Le fichier a été enregistré sous : {output_file_path}")


Index(['Animal', 'Rec', 'Trial', 'Period', 'Max_Znorm',
       'Associated Temperature', 'Stim Time', 'Age', 'Sexe', 'Weight'],
      dtype='object')
Le fichier a été enregistré sous : G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq_wo_index.xlsx


### Moyennage par animaux

In [20]:
import pandas as pd

# Charger le DataFrame contenant les valeurs d'AUC
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq_wo_index.xlsx"
df_auc_values = pd.read_excel(input_file_path)

# Calculer la moyenne des AUC par période, animal, sexe et âge
mean_auc_by_animal = df_auc_values.groupby(['Animal', 'Sexe', 'Age', 'Period'], as_index=False)['Max_Znorm'].mean()

# Définir le chemin où sauvegarder le fichier Excel des moyennes
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq_animal.xlsx"

# Sauvegarder le DataFrame des moyennes dans un fichier Excel
mean_auc_by_animal.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les moyennes d'AUC par période, sexe et âge ont été sauvegardées dans {output_file_path}")


Les moyennes d'AUC par période, sexe et âge ont été sauvegardées dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq_animal.xlsx


### Récupération des Data par animal pour period TS

In [21]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq_animal.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_thq_TS_only_animal.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où la colonne 'period' contient 'TS' (partout dans la cellule)
df_filtré = df[df['Period'].str.startswith("TS", na=False)]

# Vérifier le nombre de lignes filtrées
print(f"Nombre de lignes filtrées : {len(df_filtré)}")

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
if not df_filtré.empty:
    df_filtré.to_excel(output_file_path, index=False)
    print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")
else:
    print("Aucune ligne correspondante pour 'TS' trouvée.")

Nombre de lignes filtrées : 3
Le DataFrame filtré a été sauvegardé sous G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_thq_TS_only_animal.xlsx


## 5.2. For thq_ure

In [22]:
import pandas as pd
import numpy as np
from scipy.integrate import trapezoid

# Charger le dataset depuis le fichier Excel
file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_ure_Zscore_classic_stim.xlsx"
df = pd.read_excel(file_path)


# On inclut aussi les colonnes 'age', 'sexe', et 'weight' dans les données uniques
trials_unique = df[['animal', 'rec', 'trial', 'age', 'sexe', 'weight']].drop_duplicates().reset_index(drop=True)

# Créer une liste pour stocker les résultats
results = []

# Itérer sur chaque trial unique
for _, trial_row in trials_unique.iterrows():
    # Filtrer le DataFrame pour obtenir les données de ce trial unique
    trial_data = df[(df['animal'] == trial_row['animal']) & 
                    (df['rec'] == trial_row['rec']) & 
                    (df['trial'] == trial_row['trial'])]
    
    # Grouper les données filtrées par période
    grouped = trial_data.groupby('period')
    
    # Itérer sur chaque période ('TS', 'TB', 'PTS')
    for period, group in grouped:
        # Trouver la ligne avec la valeur maximale de Znorm
        max_row = group.loc[group['Znorm'].idxmax()]
        
        # Extraire les informations nécessaires
        Max_Znorm = max_row['Znorm']
        associated_temp = max_row['temperature']
        stim_time = max_row['Stim_Time']
        
        # Ajouter les résultats à la liste
        results.append({
            'Animal': trial_row['animal'],
            'Rec': trial_row['rec'],
            'Trial': trial_row['trial'],
            'Period': period,
            'Max_Znorm': Max_Znorm,
            'Associated Temperature': associated_temp,
            'Stim Time': stim_time,
            'Age': trial_row['age'],
            'Sexe': trial_row['sexe'],
            'Weight': trial_row['weight']
        })

# Convertir la liste de résultats en DataFrame
df_max_values = pd.DataFrame(results)

# Définir le chemin où sauvegarder le fichier Excel
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq_ure.xlsx"

# Sauvegarder le DataFrame dans un fichier Excel
df_max_values.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les données ont été sauvegardées dans {output_file_path}")


Les données ont été sauvegardées dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq_ure.xlsx


### Supression des Periods index for stat on periods

In [23]:
import pandas as pd

# Charger le fichier Excel
file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq_ure.xlsx"
df = pd.read_excel(file_path)

# Vérifier les colonnes disponibles
print(df.columns)

# Supposons que la colonne contenant les périodes s'appelle 'Period'
# Extraire les valeurs de 'Period' sans les suffixes '_index'
df['Period'] = df['Period'].str.extract(r'([A-Z]+)')[0]

# # Créer une nouvelle colonne 'Period_index' avec les indices
# df['Period_index'] = df['Period'].str.extract(r'_(\d+)')[0]

# Supprimer l'indice de la colonne 'Period'
# En utilisant str.replace pour supprimer l'indice dans 'Period'
df['Period'] = df['Period'].str.replace(r'_(\d+)', '', regex=True)

# Renommer le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq_ure_wo_index.xlsx"
df.to_excel(output_file_path, index=False)

print(f"Le fichier a été enregistré sous : {output_file_path}")


Index(['Animal', 'Rec', 'Trial', 'Period', 'Max_Znorm',
       'Associated Temperature', 'Stim Time', 'Age', 'Sexe', 'Weight'],
      dtype='object')
Le fichier a été enregistré sous : G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq_ure_wo_index.xlsx


### Moyennage par animaux

In [24]:
import pandas as pd

# Charger le DataFrame contenant les valeurs d'AUC
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq_ure_wo_index.xlsx"
df_auc_values = pd.read_excel(input_file_path)

# Calculer la moyenne des AUC par période, animal, sexe et âge
mean_auc_by_animal = df_auc_values.groupby(['Animal', 'Sexe', 'Age', 'Period'], as_index=False)['Max_Znorm'].mean()

# Définir le chemin où sauvegarder le fichier Excel des moyennes
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq_ure_animal.xlsx"

# Sauvegarder le DataFrame des moyennes dans un fichier Excel
mean_auc_by_animal.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les moyennes d'AUC par période, sexe et âge ont été sauvegardées dans {output_file_path}")


Les moyennes d'AUC par période, sexe et âge ont été sauvegardées dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq_ure_animal.xlsx


### Récupération des Data par animal pour period TS

In [25]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_by_Period_thq_ure_animal.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_thq_ure_TS_only_animal.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où la colonne 'period' contient 'TS' (partout dans la cellule)
df_filtré = df[df['Period'].str.startswith("TS", na=False)]

# Vérifier le nombre de lignes filtrées
print(f"Nombre de lignes filtrées : {len(df_filtré)}")

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
if not df_filtré.empty:
    df_filtré.to_excel(output_file_path, index=False)
    print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")
else:
    print("Aucune ligne correspondante pour 'TS' trouvée.")

Nombre de lignes filtrées : 3
Le DataFrame filtré a été sauvegardé sous G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Max_Znorm_thq_ure_TS_only_animal.xlsx


#

#

# 6. Df construction for the Mean Znorm

## 6.1. For thq stim

In [26]:
import pandas as pd

# Charger le dataset depuis le fichier Excel
file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_Zscore_classic_stim.xlsx"
df = pd.read_excel(file_path)

# Assurez-vous que la colonne 'Znorm' est de type numérique
df['Znorm'] = pd.to_numeric(df['Znorm'], errors='coerce')

# Extraire les essais uniques (en fonction de 'animal', 'rec', 'trial')
trials_unique = df[['animal', 'rec', 'trial', 'age', 'sexe', 'weight']].drop_duplicates().reset_index(drop=True)

# Créer une liste pour stocker les résultats
results = []

# Itérer sur chaque essai unique
for _, trial_row in trials_unique.iterrows():
    # Filtrer le DataFrame pour obtenir les données de cet essai unique
    trial_data = df[(df['animal'] == trial_row['animal']) & 
                    (df['rec'] == trial_row['rec']) & 
                    (df['trial'] == trial_row['trial'])]
    
    # Grouper les données filtrées par période
    grouped = trial_data.groupby('period')
    
    # Itérer sur chaque période ('TS', 'TB', 'PTS')
    for period, group in grouped:
        # Calculer la moyenne et l'écart-type de Znorm pour cette période
        mean_zscore = group['Znorm'].mean()
        
        # Ajouter les résultats à la liste
        results.append({
            'Animal': trial_row['animal'],
            'Rec': trial_row['rec'],
            'Trial': trial_row['trial'],
            'Period': period,
            'Mean_Znorm': mean_zscore,
            'Age': trial_row['age'],
            'Sexe': trial_row['sexe'],
            'Weight': trial_row['weight']
        })

# Convertir la liste de résultats en DataFrame
df_mean_sd_values = pd.DataFrame(results)

# Définir le chemin où sauvegarder le fichier Excel
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq.xlsx"

# Sauvegarder le DataFrame dans un fichier Excel
df_mean_sd_values.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les données ont été sauvegardées dans {output_file_path}")


Les données ont été sauvegardées dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq.xlsx


#### Supression des Periods index for stat on periods

In [27]:
import pandas as pd

# Charger le fichier Excel
file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq.xlsx"
df = pd.read_excel(file_path)

# Vérifier les colonnes disponibles
print(df.columns)

# Extraire les valeurs de 'Period' sans les suffixes '_index'
df['Period'] = df['Period'].str.extract(r'([A-Z]+)')[0]

# # Créer une nouvelle colonne 'Period_index' avec les indices
# df['Period_index'] = df['Period'].str.extract(r'_(\d+)')[0]

# En utilisant str.replace pour supprimer l'indice dans 'Period'
df['Period'] = df['Period'].str.replace(r'_(\d+)', '', regex=True)

# Renommer le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq_wo_index.xlsx"
df.to_excel(output_file_path, index=False)

print(f"Le fichier a été enregistré sous : {output_file_path}")


Index(['Animal', 'Rec', 'Trial', 'Period', 'Mean_Znorm', 'Age', 'Sexe',
       'Weight'],
      dtype='object')
Le fichier a été enregistré sous : G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq_wo_index.xlsx


#### Moyennage par animaux

In [28]:
import pandas as pd

# Charger le DataFrame contenant les valeurs d'AUC
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq_wo_index.xlsx"
df_auc_values = pd.read_excel(input_file_path)

# Calculer la moyenne des AUC par période, animal, sexe et âge
mean_auc_by_animal = df_auc_values.groupby(['Animal', 'Sexe', 'Age', 'Period'], as_index=False)['Mean_Znorm'].mean()

# Définir le chemin où sauvegarder le fichier Excel des moyennes
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq_animal.xlsx"

# Sauvegarder le DataFrame des moyennes dans un fichier Excel
mean_auc_by_animal.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les moyennes de mean Znorm par période, sexe et âge ont été sauvegardées dans {output_file_path}")


Les moyennes de mean Znorm par période, sexe et âge ont été sauvegardées dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq_animal.xlsx


#### Récupération des Data par animal pour period TS

In [29]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq_animal.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_thq_TS_only_animal.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où la colonne 'period' contient 'TS' (partout dans la cellule)
df_filtré = df[df['Period'].str.startswith("TS", na=False)]

# Vérifier le nombre de lignes filtrées
print(f"Nombre de lignes filtrées : {len(df_filtré)}")

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
if not df_filtré.empty:
    df_filtré.to_excel(output_file_path, index=False)
    print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")
else:
    print("Aucune ligne correspondante pour 'TS' trouvée.")

Nombre de lignes filtrées : 3
Le DataFrame filtré a été sauvegardé sous G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_thq_TS_only_animal.xlsx


## 6.2. For thq_ure

In [30]:
import pandas as pd

# Charger le dataset depuis le fichier Excel
file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_ure_Zscore_classic_stim.xlsx"
df = pd.read_excel(file_path)

# Assurez-vous que la colonne 'Znorm' est de type numérique
df['Znorm'] = pd.to_numeric(df['Znorm'], errors='coerce')

# Extraire les essais uniques (en fonction de 'animal', 'rec', 'trial')
trials_unique = df[['animal', 'rec', 'trial', 'age', 'sexe', 'weight']].drop_duplicates().reset_index(drop=True)

# Créer une liste pour stocker les résultats
results = []

# Itérer sur chaque essai unique
for _, trial_row in trials_unique.iterrows():
    # Filtrer le DataFrame pour obtenir les données de cet essai unique
    trial_data = df[(df['animal'] == trial_row['animal']) & 
                    (df['rec'] == trial_row['rec']) & 
                    (df['trial'] == trial_row['trial'])]
    
    # Grouper les données filtrées par période
    grouped = trial_data.groupby('period')
    
    # Itérer sur chaque période ('TS', 'TB', 'PTS')
    for period, group in grouped:
        # Calculer la moyenne et l'écart-type de Znorm pour cette période
        mean_zscore = group['Znorm'].mean()
        
        # Ajouter les résultats à la liste
        results.append({
            'Animal': trial_row['animal'],
            'Rec': trial_row['rec'],
            'Trial': trial_row['trial'],
            'Period': period,
            'Mean_Znorm': mean_zscore,
            'Age': trial_row['age'],
            'Sexe': trial_row['sexe'],
            'Weight': trial_row['weight']
        })

# Convertir la liste de résultats en DataFrame
df_mean_sd_values = pd.DataFrame(results)

# Définir le chemin où sauvegarder le fichier Excel
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq_ure.xlsx"

# Sauvegarder le DataFrame dans un fichier Excel
df_mean_sd_values.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les données ont été sauvegardées dans {output_file_path}")


Les données ont été sauvegardées dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq_ure.xlsx


### Supression des Periods index for stat on periods

In [31]:
import pandas as pd

# Charger le fichier Excel
file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq_ure.xlsx"
df = pd.read_excel(file_path)

# Vérifier les colonnes disponibles
print(df.columns)

# Supposons que la colonne contenant les périodes s'appelle 'Period'
# Extraire les valeurs de 'Period' sans les suffixes '_index'
df['Period'] = df['Period'].str.extract(r'([A-Z]+)')[0]

# # Créer une nouvelle colonne 'Period_index' avec les indices
# df['Period_index'] = df['Period'].str.extract(r'_(\d+)')[0]

# Supprimer l'indice de la colonne 'Period'
# En utilisant str.replace pour supprimer l'indice dans 'Period'
df['Period'] = df['Period'].str.replace(r'_(\d+)', '', regex=True)

# Renommer le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq_ure_wo_index.xlsx"
df.to_excel(output_file_path, index=False)

print(f"Le fichier a été enregistré sous : {output_file_path}")


Index(['Animal', 'Rec', 'Trial', 'Period', 'Mean_Znorm', 'Age', 'Sexe',
       'Weight'],
      dtype='object')
Le fichier a été enregistré sous : G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq_ure_wo_index.xlsx


### Moyennage par animaux

In [32]:
import pandas as pd

# Charger le DataFrame contenant les valeurs d'AUC
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq_ure_wo_index.xlsx"
df_auc_values = pd.read_excel(input_file_path)

# Calculer la moyenne des AUC par période, animal, sexe et âge
mean_auc_by_animal = df_auc_values.groupby(['Animal', 'Sexe', 'Age', 'Period'], as_index=False)['Mean_Znorm'].mean()

# Définir le chemin où sauvegarder le fichier Excel des moyennes
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq_ure_animal.xlsx"

# Sauvegarder le DataFrame des moyennes dans un fichier Excel
mean_auc_by_animal.to_excel(output_file_path, index=False)

# Afficher un message de confirmation
print(f"Les moyennes de mean Znorm par période, sexe et âge ont été sauvegardées dans {output_file_path}")


Les moyennes de mean Znorm par période, sexe et âge ont été sauvegardées dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq_ure_animal.xlsx


### Récupération des Data par animal pour period TS

In [33]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_by_Period_thq_ure_animal.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_thq_ure_TS_only_animal.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où la colonne 'period' contient 'TS' (partout dans la cellule)
df_filtré = df[df['Period'].str.startswith("TS", na=False)]

# Vérifier le nombre de lignes filtrées
print(f"Nombre de lignes filtrées : {len(df_filtré)}")

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
if not df_filtré.empty:
    df_filtré.to_excel(output_file_path, index=False)
    print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")
else:
    print("Aucune ligne correspondante pour 'TS' trouvée.")

Nombre de lignes filtrées : 3
Le DataFrame filtré a été sauvegardé sous G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Quantif_by_Periods/Mean_Znorm_thq_ure_TS_only_animal.xlsx


#

#

## 7.1. Creation of the directory for saving

In [2]:
import os

# Définir le chemin de base et le répertoire à créer
base_dir = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2"
save_dir = os.path.join(base_dir, "Thresholding_Thq_vs_Ure")

# Vérifier si le répertoire existe, sinon le créer
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"Le répertoire '{save_dir}' a été créé avec succès.")
else:
    print(f"Le répertoire '{save_dir}' existe déjà.")

Le répertoire 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2\Thresholding_Thq_vs_Ure' existe déjà.


## 7.2. Recuperation of Threshold value

In [1]:
import pandas as pd


# Chemin vers le fichier des seuils (mean_max_z_score)
Threshold_file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_shuffling_threshold/global_distribution/Threshold_value.xlsx'

# Lire les données des seuils pour chaque animal et rec
df_threshold = pd.read_excel(Threshold_file_path)

# Obtenir la valmeur de Threshold
Threshold = df_threshold['Value'].values

print (Threshold)

[1.85966108]


## 7.3. For cool stim

#### New DF witout TS for classic Z

In [38]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_Zscore_classic.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_Zscore_classic_wo_TS.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où 'period' ne commence pas par "TS_"
df_filtered = df[~df['period'].str.startswith("TS_", na=False)]

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
df_filtered.to_excel(output_file_path, index=False)

print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")


Le DataFrame filtré a été sauvegardé sous G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_Zscore_classic_wo_TS.xlsx


#### Binarisation of response rate with Znorm and unique threshold

In [51]:
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_Zscore_classic_stim.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
response_rate_results = []

# Définir le seuil fixe
threshold = Threshold

# Liste des animaux
animals = df['animal'].unique()

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Filtrer les lignes qui commencent par "TS_"
            df_ts = df_trial[df_trial['period'].str.startswith("TS_")]
            
            # Vérifier s'il y a des données dans la période "TS"
            if not df_ts.empty:
                # Lire les valeurs de Z-score dans la période "TS"
                z_scores_ts = df_ts['Znorm'].values
                
                # Vérifier si au moins une valeur de Z-score dépasse le seuil
                response_rate = 1 if (z_scores_ts > threshold).any() else 0
                
                # Ajouter les résultats à la liste
                response_rate_results.append({
                    'animal': animal,
                    'rec': rec,
                    'trial': trial,
                    'age': age,    # Ajout de la colonne 'age'
                    'sexe': sexe,  # Ajout de la colonne 'sexe'
                    'response_rate': response_rate
                })

# Convertir les résultats en DataFrame
response_rate_df = pd.DataFrame(response_rate_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Thresholding_Thq_vs_Ure/Thq_responsive_rate.xlsx'
response_rate_df.to_excel(output_file_path, index=False)

# Afficher les résultats
print(response_rate_df)


        animal        rec    trial age sexe  response_rate
0   2023.10.06  Thermic_1  Trial_1  P7    M              1
1   2023.10.06  Thermic_1  Trial_2  P7    M              1
2   2023.10.06  Thermic_1  Trial_3  P7    M              1
3   2023.10.06  Thermic_1  Trial_4  P7    M              1
4   2023.10.06  Thermic_1  Trial_5  P7    M              1
5   2023.10.06  Thermic_1  Trial_6  P7    M              1
6   2023.10.06  Thermic_1  Trial_7  P7    M              1
7   2023.10.06  Thermic_1  Trial_8  P7    M              1
8   2023.10.06  Thermic_2  Trial_1  P7    M              1
9   2023.10.06  Thermic_2  Trial_2  P7    M              1
10  2023.10.06  Thermic_2  Trial_3  P7    M              1
11  2023.10.06  Thermic_2  Trial_4  P7    M              1
12  2023.10.06  Thermic_2  Trial_5  P7    M              0
13  2023.10.06  Thermic_2  Trial_6  P7    M              0
14  2023.10.06  Thermic_2  Trial_7  P7    M              1
15  2023.10.06  Thermic_2  Trial_8  P7    M             

#### Coding the Responsive Rate for each period and trial

In [43]:
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_Zscore_classic_wo_TS.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
response_rate_results = []

# Définir le seuil fixe
threshold = Threshold

# Liste des animaux
animals = df['animal'].unique()

# Liste des périodes à analyser
periods = ['TB', 'TS', 'PTS']

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Parcourir les périodes (TB, TS, PTS)
            for period in periods:
                # Filtrer les données pour la période actuelle
                df_period = df_trial[df_trial['period'].str.startswith(period)]
                
                # Vérifier s'il y a des données dans cette période
                if not df_period.empty:
                    # Lire les valeurs de Z-score dans la période
                    z_scores_period = df_period['Znorm'].values
                    
                    # Vérifier si au moins une valeur de Z-score dépasse le seuil
                    response_rate = 1 if (z_scores_period > threshold).any() else 0
                    
                    # Ajouter les résultats à la liste
                    response_rate_results.append({
                        'animal': animal,
                        'rec': rec,
                        'trial': trial,
                        'period': period,  # Ajout de la période
                        'age': age,    # Ajout de la colonne 'age'
                        'sexe': sexe,  # Ajout de la colonne 'sexe'
                        'response_rate': response_rate
                    })

# Convertir les résultats en DataFrame
response_rate_df = pd.DataFrame(response_rate_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Thresholding_Thq_vs_Ure/Thq_responsive_rate_by_period.xlsx'
response_rate_df.to_excel(output_file_path, index=False)

# Afficher les résultats
print(response_rate_df)


        animal        rec    trial period age sexe  response_rate
0   2023.10.06  Thermic_1  Trial_1     TB  P7    M              0
1   2023.10.06  Thermic_1  Trial_1    PTS  P7    M              0
2   2023.10.06  Thermic_1  Trial_2     TB  P7    M              1
3   2023.10.06  Thermic_1  Trial_2    PTS  P7    M              1
4   2023.10.06  Thermic_1  Trial_3     TB  P7    M              0
..         ...        ...      ...    ...  ..  ...            ...
91  2023.10.17  Thermic_5  Trial_6    PTS  P7    F              1
92  2023.10.17  Thermic_5  Trial_7     TB  P7    F              1
93  2023.10.17  Thermic_5  Trial_7    PTS  P7    F              1
94  2023.10.17  Thermic_5  Trial_8     TB  P7    F              1
95  2023.10.17  Thermic_5  Trial_8    PTS  P7    F              1

[96 rows x 7 columns]


#### Coding the Normalized response rate

In [44]:
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_Zscore_classic_stim.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
norm_response_rate_results = []

# Liste des animaux
animals = df['animal'].unique()

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Parcourir chaque période du trial
            for period in df_trial['period'].unique():
                # Filtrer les données pour la période actuelle
                df_period = df_trial[df_trial['period'] == period]
                
                # Lire les valeurs de Z-score pour la période
                z_scores_period = df_period['Znorm'].values
                
                # Définir un seuil fixe de 1.91 (au lieu de mean_95_percent)
                threshold = Threshold
                
                # Calculer le nombre de Z-scores supérieurs au seuil
                numb_of_sup_Z = (z_scores_period > threshold).sum()
                
                # Calculer le nombre total de Z-scores dans la période
                numb_of_tot_Z = len(z_scores_period)
                
                # Calculer le taux de réponse normalisé
                Norm_response_rate = (numb_of_sup_Z / numb_of_tot_Z) * 100 if numb_of_tot_Z > 0 else 0
                
                # Ajouter les résultats à la liste
                norm_response_rate_results.append({
                    'animal': animal,
                    'rec': rec,
                    'trial': trial,
                    'period': period,
                    'age': age,    # Ajout de la colonne 'age'
                    'sexe': sexe,  # Ajout de la colonne 'sexe'
                    'numb_of_sup_Z': numb_of_sup_Z,
                    'numb_of_tot_Z': numb_of_tot_Z,
                    'Norm_response_rate': Norm_response_rate
                })

# Convertir les résultats en DataFrame
norm_response_rate_df = pd.DataFrame(norm_response_rate_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Thresholding_Thq_vs_Ure/Thq_Norm_response_rate.xlsx'
norm_response_rate_df.to_excel(output_file_path, index=False)

print(f"Les résultats ont été sauvegardés dans {output_file_path}")

# # Afficher les résultats
# print(norm_response_rate_df)


Les résultats ont été sauvegardés dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Thresholding_Thq_vs_Ure/Thq_Norm_response_rate.xlsx


#### Coding the Magnitude of the response

In [3]:
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_Zscore_classic.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
response_magnitude_results = []

# Liste des animaux
animals = df['animal'].unique()

# Définir le seuil pour les Z-scores
threshold = Threshold

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Parcourir chaque période du trial
            for period in df_trial['period'].unique():
                # Filtrer les données pour la période actuelle
                df_period = df_trial[df_trial['period'] == period]
                
                # Lire les valeurs de Z-score pour la période
                z_scores_period = df_period['Znorm'].values
                
                # Filtrer les Z-scores au-dessus du seuil
                z_scores_above_threshold = z_scores_period[z_scores_period > threshold]
                
                # Calculer la magnitude de la réponse normalisée
                # Diviser la somme des Z-scores au-dessus du seuil par le nombre total de valeurs dans la période
                if len(z_scores_period) > 0:  # Éviter la division par zéro
                    response_magnitude = z_scores_above_threshold.sum() / len(z_scores_period)
                else:
                    response_magnitude = 0  # Si la période est vide, définir à 0
                
                # Ajouter les résultats à la liste
                response_magnitude_results.append({
                    'animal': animal,
                    'rec': rec,
                    'trial': trial,
                    'period': period,
                    'age': age,    # Ajout de la colonne 'age'
                    'sexe': sexe,  # Ajout de la colonne 'sexe'
                    'response_magnitude': response_magnitude
                })

# Convertir les résultats en DataFrame
response_magnitude_df = pd.DataFrame(response_magnitude_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Thresholding_Thq_vs_Ure/Thq_Response_Magnitude.xlsx'
response_magnitude_df.to_excel(output_file_path, index=False)

print(f"Les résultats ont été sauvegardés dans {output_file_path}")

# Afficher un aperçu des résultats
print(response_magnitude_df)


Les résultats ont été sauvegardés dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Thresholding_Thq_vs_Ure/Thq_Response_Magnitude.xlsx
         animal        rec    trial  period age sexe  response_magnitude
0    2023.10.06  Thermic_1  Trial_1  ITST_1  P7    M            0.000000
1    2023.10.06  Thermic_1  Trial_1    TB_1  P7    M            0.000000
2    2023.10.06  Thermic_1  Trial_1    TS_1  P7    M            0.487692
3    2023.10.06  Thermic_1  Trial_1   PTS_1  P7    M            0.000000
4    2023.10.06  Thermic_1  Trial_1  ITST_2  P7    M            0.000000
..          ...        ...      ...     ...  ..  ...                 ...
193  2023.10.17  Thermic_5  Trial_7  ITST_8  P7    F            0.000000
194  2023.10.17  Thermic_5  Trial_8    TB_8  P7    F            0.092386
195  2023.10.17  Thermic_5  Trial_8    TS_8  P7    F            0.000000
196  2023.10.17  Thermic_5  Trial_8   PTS_8  P7    F            0.059265
197  2023.10.17  Thermic_

## 7.4. For Thq_ure stimulation

#### New DF witout TS for classic Z

In [45]:
import pandas as pd

# Chemin du fichier d'origine
input_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_ure_Zscore_classic.xlsx"

# Chemin pour le fichier de sortie
output_file_path = "G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_ure_Zscore_classic_wo_TS.xlsx"

# Charger le DataFrame depuis le fichier Excel
df = pd.read_excel(input_file_path)

# Filtrer les lignes où 'period' ne commence pas par "TS_"
df_filtered = df[~df['period'].str.startswith("TS_", na=False)]

# Sauvegarder le DataFrame filtré dans un nouveau fichier Excel
df_filtered.to_excel(output_file_path, index=False)

print(f"Le DataFrame filtré a été sauvegardé sous {output_file_path}")


Le DataFrame filtré a été sauvegardé sous G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_ure_Zscore_classic_wo_TS.xlsx


#### Coding of Responsive Rate for each unique trials over TS period

In [46]:
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_ure_Zscore_classic_stim.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
response_rate_results = []

# Liste des animaux
animals = df['animal'].unique()

# Définition du seuil fixe
threshold = Threshold

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Filtrer les lignes qui commencent par "TS_"
            df_ts = df_trial[df_trial['period'].str.startswith("TS_")]
            
            # Vérifier s'il y a des données dans la période "TS"
            if not df_ts.empty:
                # Lire les valeurs de Z-score dans la période "TS"
                z_scores_ts = df_ts['Znorm'].values
                
                # Vérifier si au moins une valeur de Z-score dépasse le seuil fixe
                response_rate = 1 if (z_scores_ts > threshold).any() else 0
                
                # Ajouter les résultats à la liste
                response_rate_results.append({
                    'animal': animal,
                    'rec': rec,
                    'trial': trial,
                    'age': age,    # Ajout de la colonne 'age'
                    'sexe': sexe,  # Ajout de la colonne 'sexe'
                    'response_rate': response_rate
                })

# Convertir les résultats en DataFrame
response_rate_df = pd.DataFrame(response_rate_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Thresholding_Thq_vs_Ure/Thq_Ure_responsive_rate.xlsx'
response_rate_df.to_excel(output_file_path, index=False)

# Afficher les résultats
print(response_rate_df)


        animal            rec    trial age sexe  response_rate
0   2023.10.06  urethane_Thc1  Trial_1  P7    M              1
1   2023.10.06  urethane_Thc1  Trial_2  P7    M              1
2   2023.10.06  urethane_Thc1  Trial_3  P7    M              1
3   2023.10.06  urethane_Thc1  Trial_4  P7    M              1
4   2023.10.06  urethane_Thc1  Trial_5  P7    M              0
5   2023.10.06  urethane_Thc1  Trial_6  P7    M              1
6   2023.10.06  urethane_Thc1  Trial_7  P7    M              1
7   2023.10.06  urethane_Thc1  Trial_8  P7    M              1
8   2023.10.06  urethane_Thc2  Trial_1  P7    M              1
9   2023.10.06  urethane_Thc2  Trial_2  P7    M              1
10  2023.10.06  urethane_Thc2  Trial_3  P7    M              1
11  2023.10.06  urethane_Thc2  Trial_4  P7    M              0
12  2023.10.06  urethane_Thc2  Trial_5  P7    M              1
13  2023.10.06  urethane_Thc2  Trial_6  P7    M              1
14  2023.10.06  urethane_Thc2  Trial_7  P7    M        

#### Coding the Responsive Rate for each period and trial

In [47]:
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_ure_Zscore_classic_stim.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
response_rate_results = []

# Liste des animaux
animals = df['animal'].unique()

# Liste des périodes à analyser
periods = ['TB', 'TS', 'PTS']

# Seuil fixe
threshold = Threshold

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Parcourir les périodes (TB, TS, PTS)
            for period in periods:
                # Filtrer les données pour la période actuelle
                df_period = df_trial[df_trial['period'].str.startswith(period)]
                
                # Vérifier s'il y a des données dans cette période
                if not df_period.empty:
                    # Lire les valeurs de Z-score dans la période
                    z_scores_period = df_period['Znorm'].values
                    
                    # Vérifier si au moins une valeur de Z-score dépasse le seuil
                    response_rate = 1 if (z_scores_period > threshold).any() else 0
                    
                    # Ajouter les résultats à la liste
                    response_rate_results.append({
                        'animal': animal,
                        'rec': rec,
                        'trial': trial,
                        'period': period,  # Ajout de la période
                        'age': age,    # Ajout de la colonne 'age'
                        'sexe': sexe,  # Ajout de la colonne 'sexe'
                        'response_rate': response_rate
                    })

# Convertir les résultats en DataFrame
response_rate_df = pd.DataFrame(response_rate_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Thresholding_Thq_vs_Ure/Thq_Ure_responsive_rate_by_period.xlsx'
response_rate_df.to_excel(output_file_path, index=False)

# Afficher les résultats
print(response_rate_df)


         animal            rec    trial period age sexe  response_rate
0    2023.10.06  urethane_Thc1  Trial_1     TB  P7    M              1
1    2023.10.06  urethane_Thc1  Trial_1     TS  P7    M              1
2    2023.10.06  urethane_Thc1  Trial_1    PTS  P7    M              1
3    2023.10.06  urethane_Thc1  Trial_2     TB  P7    M              1
4    2023.10.06  urethane_Thc1  Trial_2     TS  P7    M              1
..          ...            ...      ...    ...  ..  ...            ...
139  2023.10.17  urethane_Thc2  Trial_7     TS  P7    F              1
140  2023.10.17  urethane_Thc2  Trial_7    PTS  P7    F              1
141  2023.10.17  urethane_Thc2  Trial_8     TB  P7    F              0
142  2023.10.17  urethane_Thc2  Trial_8     TS  P7    F              0
143  2023.10.17  urethane_Thc2  Trial_8    PTS  P7    F              0

[144 rows x 7 columns]


#### Coding the Normalized response rate

In [49]:
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_ure_Zscore_classic_stim.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
norm_response_rate_results = []

# Liste des animaux
animals = df['animal'].unique()

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Parcourir chaque période du trial
            for period in df_trial['period'].unique():
                # Filtrer les données pour la période actuelle
                df_period = df_trial[df_trial['period'] == period]
                
                # Lire les valeurs de Z-score pour la période
                z_scores_period = df_period['Znorm'].values
                
                # Définir le seuil à 1.91
                threshold = Threshold
                
                # Calculer le nombre de Z-scores supérieurs au seuil
                numb_of_sup_Z = (z_scores_period > threshold).sum()
                
                # Calculer le nombre total de Z-scores dans la période
                numb_of_tot_Z = len(z_scores_period)
                
                # Calculer le taux de réponse normalisé
                Norm_response_rate = (numb_of_sup_Z / numb_of_tot_Z) * 100 if numb_of_tot_Z > 0 else 0
                
                # Ajouter les résultats à la liste
                norm_response_rate_results.append({
                    'animal': animal,
                    'rec': rec,
                    'trial': trial,
                    'period': period,
                    'age': age,    # Ajout de la colonne 'age'
                    'sexe': sexe,  # Ajout de la colonne 'sexe'
                    'numb_of_sup_Z': numb_of_sup_Z,
                    'numb_of_tot_Z': numb_of_tot_Z,
                    'Norm_response_rate': Norm_response_rate
                })

# Convertir les résultats en DataFrame
norm_response_rate_df = pd.DataFrame(norm_response_rate_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Thresholding_Thq_vs_Ure/Thq_Ure_Norm_response_rate.xlsx'
norm_response_rate_df.to_excel(output_file_path, index=False)

print(f"Les résultats ont été sauvegardés dans {output_file_path}")

# # Afficher les résultats
# print(norm_response_rate_df)


Les résultats ont été sauvegardés dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Thresholding_Thq_vs_Ure/Thq_Ure_Norm_response_rate.xlsx


#### Coding the Magnitude of the response

In [4]:
import pandas as pd

# Chemin vers le fichier des Z-scores classiques
file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Df_thq_ure_Zscore_classic.xlsx'

# Lire les données des Z-scores classiques
df = pd.read_excel(file_path)

# Conversion de Stim_Time de ms à s (si nécessaire)
df['Stim_Time'] = df['Stim_Time'] / 1000

# Initialiser une liste pour stocker les résultats
response_magnitude_results = []

# Liste des animaux
animals = df['animal'].unique()

# Définir le seuil pour les Z-scores
threshold = Threshold

# Parcourir chaque animal
for animal in animals:
    # Filtrer les données pour l'animal en cours
    df_animal = df[df['animal'] == animal]
    
    # Récupérer l'âge et le sexe de l'animal
    age = df_animal['age'].iloc[0]  # Suppose que la colonne 'age' existe
    sexe = df_animal['sexe'].iloc[0]  # Suppose que la colonne 'sexe' existe

    # Parcourir chaque rec
    for rec in df_animal['rec'].unique():
        # Filtrer les données pour le rec actuel
        df_rec = df_animal[df_animal['rec'] == rec]
        
        # Parcourir chaque trial
        for trial in df_rec['trial'].unique():
            # Filtrer les données pour le trial actuel
            df_trial = df_rec[df_rec['trial'] == trial]
            
            # Parcourir chaque période du trial
            for period in df_trial['period'].unique():
                # Filtrer les données pour la période actuelle
                df_period = df_trial[df_trial['period'] == period]
                
                # Lire les valeurs de Z-score pour la période
                z_scores_period = df_period['Znorm'].values
                
                # Filtrer les Z-scores au-dessus du seuil
                z_scores_above_threshold = z_scores_period[z_scores_period > threshold]
                
                # Calculer la magnitude de la réponse normalisée
                # Diviser la somme des Z-scores au-dessus du seuil par le nombre total de valeurs dans la période
                if len(z_scores_period) > 0:  # Éviter la division par zéro
                    response_magnitude = z_scores_above_threshold.sum() / len(z_scores_period)
                else:
                    response_magnitude = 0  # Si la période est vide, définir à 0
                
                # Ajouter les résultats à la liste
                response_magnitude_results.append({
                    'animal': animal,
                    'rec': rec,
                    'trial': trial,
                    'period': period,
                    'age': age,    # Ajout de la colonne 'age'
                    'sexe': sexe,  # Ajout de la colonne 'sexe'
                    'response_magnitude': response_magnitude
                })

# Convertir les résultats en DataFrame
response_magnitude_df = pd.DataFrame(response_magnitude_results)

# Sauvegarder les résultats dans un fichier Excel
output_file_path = 'G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Thresholding_Thq_vs_Ure/Thq_ure_Response_Magnitude.xlsx'
response_magnitude_df.to_excel(output_file_path, index=False)

print(f"Les résultats ont été sauvegardés dans {output_file_path}")

# Afficher un aperçu des résultats
print(response_magnitude_df)


Les résultats ont été sauvegardés dans G:/PhD/Experimentation/Calcium_imaging_WF/analysis_DFF/raw_data_Thq_vs_Thq_ure_2/Thresholding_Thq_vs_Ure/Thq_ure_Response_Magnitude.xlsx
         animal            rec    trial  period age sexe  response_magnitude
0    2023.10.06  urethane_Thc1  Trial_1  ITST_1  P7    M            0.000000
1    2023.10.06  urethane_Thc1  Trial_1    TB_1  P7    M            0.076093
2    2023.10.06  urethane_Thc1  Trial_1    TS_1  P7    M            2.198682
3    2023.10.06  urethane_Thc1  Trial_1   PTS_1  P7    M            0.689051
4    2023.10.06  urethane_Thc1  Trial_1  ITST_2  P7    M            0.000000
..          ...            ...      ...     ...  ..  ...                 ...
193  2023.10.17  urethane_Thc2  Trial_7  ITST_8  P7    F            0.000000
194  2023.10.17  urethane_Thc2  Trial_8    TB_8  P7    F            0.000000
195  2023.10.17  urethane_Thc2  Trial_8    TS_8  P7    F            0.000000
196  2023.10.17  urethane_Thc2  Trial_8   PTS_8  P7   